# Introduction
Process mining helps uncover insights from data that represent different events in a process. It is valuable for understanding workflows, finding inefficiencies, and optimizing development processes. Using Kaiaulu  for process mining enables this discovery for Github Events (e.g. creating issues, push events, deleting branches, etc). Allowing users to visualize and analyze event data effectively. This notebook demonstrates how to use Kaiaulu to download and parse Github Event data. As well how to interact with the data to leverage process discovery for deeper analysis. This [video](https://www.youtube.com/watch?v=XLHtvt36g6U&t=1181s) provides an overview of process mining concepts and its applications and can be used as a reference point to begin. This notebook heavily relies on the python package [pm4py](https://github.com/process-intelligence-solutions/pm4py) to create event mappings.

### Requirements

-   Gihub Access Token 
-   [Kaiaulu](https://github.com/sailuh/kaiaulu) R package to download data via CLI
-   Python environment with [pm4py](https://github.com/process-intelligence-solutions/pm4py) installed


In [2]:
# Python Imports 

import argparse
import pandas
import os
from datetime import datetime
import pm4py
import subprocess
import yaml

### Project Configuration File

To streamline downloading and parsing data for Github Events we will specify a configuration file instead of entering parameters manually for both R and python environments. We can use the Kaiaulu.yml configuration file as a template. The configuration file content that interests us is the `issue_event` download path.

```yaml
github:
    project_key_1:
      # Obtained from the project's GitHub URL
      owner: sailuh
      repo: kaiaulu
      # Download using `download_github_comments.Rmd`
      issue_or_pr_comment: ../../rawdata/kaiaulu/github/sailuh_kaiaulu/issue_or_pr_comment/
      issue: ../../rawdata/kaiaulu/github/sailuh_kaiaulu/issue/
      issue_search: ../../rawdata/kaiaulu/github/sailuh_kaiaulu/issue_search/
      issue_event: ../rawdata/kaiaulu/github/sailuh_kaiaulu/issue_event/
      pull_request: ../../rawdata/kaiaulu/github/sailuh_kaiaulu/pull_request/
      commit: ../../rawdata/kaiaulu/github/sailuh_kaiaulu/commit/


### Download and Parse data with Kaiaulu ghevents.R (CLI)


In [11]:
# Save and set the current working directory to Kaiaulu
cwd = os.getcwd()
os.chdir(os.path.expanduser("~/Desktop/Kaiaulu/Working_issues/kaiaulu/"))

# To download use the download command specifing the <config_file> <github_token>

command = f"Rscript exec/ghevents.R download conf/process_mining.yml --token_path=~/.ssh/github_token"
subprocess.run(command, shell=True, check=True)


ℹ Downloading GitHub Project Issue Events using R/github.R
Latest date:1740475673
Oldest date:1733730684
File name: ../rawdata/kaiaulu/sailuh_kaiaulu/issue_event/_1733730684_1740475673.json
extracted dates for page 1
Written to file: ../rawdata/kaiaulu/sailuh_kaiaulu/issue_event/sailuh_kaiaulu_1733730684_1740475673.json
Latest date:1733730684
Oldest date:1733658435
File name: ../rawdata/kaiaulu/sailuh_kaiaulu/issue_event/_1733658435_1733730684.json
extracted dates for page 2
Written to file: ../rawdata/kaiaulu/sailuh_kaiaulu/issue_event/sailuh_kaiaulu_1733658435_1733730684.json
Latest date:1733658435
Oldest date:1731893713
File name: ../rawdata/kaiaulu/sailuh_kaiaulu/issue_event/_1731893713_1733658435.json
extracted dates for page 3
Written to file: ../rawdata/kaiaulu/sailuh_kaiaulu/issue_event/sailuh_kaiaulu_1731893713_1733658435.json
Latest date:1731889586
Oldest date:1731360656
File name: ../rawdata/kaiaulu/sailuh_kaiaulu/issue_event/_1731360656_1731889586.json
extracted dates for p

KeyboardInterrupt: 

In [10]:
# To Parse use the parse command and specify the <config_path> <output_dir>

command = f"Rscript exec/ghevents.R parse conf/kaiaulu.yml ../rawdata/output.csv"
subprocess.run(command, shell=True, check=True)

ℹ Parsing Github issue events from ../rawdata/kaiaulu/sailuh_kaiaulu/issue_event/
ℹ Parsed data saved to ../rawdata/output.csv


CompletedProcess(args='Rscript exec/ghevents.R parse conf/kaiaulu.yml ../rawdata/output.csv', returncode=0)

### Get Start and End Activies From Event Logs

In [17]:
# Specify the csv file path <csv_path>
csv_path = "../rawdata/output.csv"

def start_end_activities(csv_path): 
    event_log = pandas.read_csv(csv_path)
    event_log = pm4py.format_dataframe(event_log, case_id='id', activity_key='event', timestamp_key='created_at')
    start_activities = pm4py.get_start_activities(event_log)
    end_activities = pm4py.get_end_activities(event_log)
    return start_activities, end_activities

print(start_end_activities(csv_path))


({'renamed': 84, 'referenced': 778, 'closed': 244, 'labeled': 385, 'assigned': 391, 'mentioned': 634, 'subscribed': 630, 'merged': 38, 'unassigned': 56, 'head_ref_force_pushed': 25, 'review_requested': 39, 'convert_to_draft': 1, 'unsubscribed': 2, 'head_ref_deleted': 8, 'locked': 1, 'converted_to_discussion': 1, 'review_request_removed': 3, 'connected': 22, 'added_to_project_v2': 43, 'ready_for_review': 1, 'project_v2_item_status_changed': 67, 'milestoned': 96, 'demilestoned': 6, 'sub_issue_added': 1, 'parent_issue_added': 1, 'sub_issue_removed': 1, 'parent_issue_removed': 1, 'reopened': 5, 'unlabeled': 9, 'pinned': 3}, {'renamed': 84, 'referenced': 778, 'closed': 244, 'labeled': 385, 'assigned': 391, 'mentioned': 634, 'subscribed': 630, 'merged': 38, 'unassigned': 56, 'head_ref_force_pushed': 25, 'review_requested': 39, 'convert_to_draft': 1, 'unsubscribed': 2, 'head_ref_deleted': 8, 'locked': 1, 'converted_to_discussion': 1, 'review_request_removed': 3, 'connected': 22, 'added_to_pro

### Generate Tree from Event Logs

In [19]:
# Specify a output_dir where the images will be sent, if it doesnt exist it will be created
output_dir = "../rawdata/images/"

def generate_tree(csv_path, output_dir):
    event_log = pandas.read_csv(csv_path)
    event_log = pm4py.format_dataframe(event_log, case_id='issue_number', activity_key='event', timestamp_key='created_at')

    # Create if folder doesn't exist
    output_dir = os.path.expanduser(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # Set up path and name of .png
    output_dir = os.path.expanduser(output_dir)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_name = f"process_tree_{timestamp}.png"
    output_path = os.path.join(output_dir, file_name)

    process_tree = pm4py.discover_process_tree_inductive(event_log)
    pm4py.save_vis_process_tree(process_tree, output_path)
    print("Tree generated and saved")

generate_tree(csv_path, output_dir)

Tree generated and saved


### Generate Graph from Event Logs

In [20]:
# Pass the same arguments as trees

def generate_graph(csv_path, output_dir):
    event_log = pandas.read_csv(csv_path)
    event_log = pm4py.format_dataframe(event_log, case_id='issue_number', activity_key='event', timestamp_key='created_at')
    
    # Create if folder doesn't exist
    output_dir = os.path.expanduser(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    # Set up path and name of .png
    output_dir = os.path.expanduser(output_dir)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_name = f"process_graph_{timestamp}.png"
    output_path = os.path.join(output_dir, file_name)

    process_tree = pm4py.discover_process_tree_inductive(event_log)
    bpmn_model = pm4py.convert_to_bpmn(process_tree)
    pm4py.save_vis_bpmn(bpmn_model, output_path, format="png")
    print("Graph generated and saved")

generate_graph(csv_path, output_dir)

Graph generated and saved
